# Fig S11b,c — marginal and COJO joint P of credible-set representatives (data processing)

- **S11b:** marginal P of the top-PIP variant of every SuSiE-rRSS credible set (12 AD GWAS, eb_mix export), regions with one vs. multiple credible sets. Also prints the counts for Results l.327–330, which use the smallest marginal P in each credible set.
- **S11c:** COJO joint P (`24` `[cojo]`, job `cojo_multi_cs`: top-PIP variant of every credible set in each multi-CS region, fitted jointly, no P filter). Prints the counts for Results l.331–334, and keeps the representatives whose marginal P > 5 × 10⁻⁸ for the joint-P distribution and the marginal → joint plot.

P values are computed from z on the log scale (no underflow for very strong signals).

**Outputs:** `figure_data/FigS12B.AD_GWAS_CS_Top_Variant_Marginal_P`, `Fig6FG.AD_GWAS_COJO_Joint`, `Fig6FG.AD_GWAS_COJO_Joint_SubGWS`

In [1]:
source("../../R/utils.R")

fm_root  <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10"
cojo_res <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO/cojo_multi_cs/result"

## -log10 two-sided P from z, without underflow for very large |z|
mlog10p <- function(z) -(log(2) + pnorm(-abs(z), log.p = TRUE)) / log(10)
p_marks <- c("5e-8" = 5e-8, "1e-5" = 1e-5, "1e-4" = 1e-4)

## S11b: marginal P of the top variant in each credible set

In [2]:
## ---- top-PIP variant of every SuSiE-rRSS credible set, all 12 datasets ----------
eb_mix <- read_gwas_tables(file.path(fm_root, "export/")) %>%
  filter(!is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("", "0", "susie_rss_0"))

top_var <- eb_mix %>%
  group_by(study, block_id, cs = cs_coverage_0.95) %>%
  mutate(cs_min_mlog10p = max(mlog10p(z))) %>%                  # smallest marginal P in the CS
  slice_max(PIP, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  group_by(study, block_id) %>%
  mutate(n_cs = n_distinct(cs)) %>%
  ungroup() %>%
  transmute(study, study_label = unname(cohort_labels[study]), block_id, cs, variant_ID, PIP, z,
            marginal_mlog10p = mlog10p(z), cs_min_mlog10p,
            region = factor(if_else(n_cs > 1, "Multiple credible sets", "One credible set"),
                            levels = c("One credible set", "Multiple credible sets")))

## counts at the marks (Results l.327-330 use the smallest marginal P in each CS)
top_var %>%
  summarise(n_cs = n(),
            pct_cs_gws  = round(100 * mean(cs_min_mlog10p > -log10(5e-8)), 1),
            pct_cs_1e5  = round(100 * mean(cs_min_mlog10p > -log10(1e-5)), 1),
            n_cs_min_p_above_1e4 = sum(cs_min_mlog10p < -log10(1e-4)),
            n_of_those_in_multi  = sum(cs_min_mlog10p < -log10(1e-4) & region == "Multiple credible sets"))
top_var %>% group_by(region) %>%
  summarise(n_cs = n(), top_var_below_gws = sum(marginal_mlog10p < -log10(5e-8)), .groups = "drop")

save_data(top_var, "FigS12B.AD_GWAS_CS_Top_Variant_Marginal_P")

n_cs,pct_cs_gws,pct_cs_1e5,n_cs_min_p_above_1e4,n_of_those_in_multi
<int>,<dbl>,<dbl>,<int>,<int>
403,80.1,97,5,5


region,n_cs,top_var_below_gws
<fct>,<int>,<int>
One credible set,282,55
Multiple credible sets,121,26


## S11c: COJO joint P in multi-CS regions

In [3]:
## ---- COJO joint model in the multi-CS regions (24 [cojo], cojo_multi_cs) ----------
## one row per CS: its top-PIP variant, fitted jointly with the other CSs' top-PIP variants
cojo <- list.files(cojo_res, pattern = "\\.cojo_joint\\.tsv$", recursive = TRUE, full.names = TRUE) %>%
  map_dfr(read_tsv, show_col_types = FALSE) %>%
  mutate(marginal_mlog10p = mlog10p(marginal_z),
         joint_mlog10p    = mlog10p(joint_z),
         study_label      = unname(cohort_labels[study])) %>%
  left_join(select(top_var, study, block_id, cs, cs_min_mlog10p), by = c("study", "block_id", "cs"))

## coverage: study x region pairs with a joint fit, and representatives with a joint P
tibble(n_pairs = n_distinct(paste(cojo$study, cojo$block_id)), n_cs = nrow(cojo),
       n_joint_missing = sum(is.na(cojo$joint_z)))

## Results l.331-334 (Fig S11c)
cojo %>% group_by(study, block_id) %>%
  summarise(all_joint_1e5 = all(joint_mlog10p > -log10(1e-5), na.rm = FALSE), .groups = "drop") %>%
  summarise(n_pairs = n(), n_all_joint_below_1e5 = sum(all_joint_1e5 %in% TRUE))
weak <- cojo %>% filter(cs_min_mlog10p < -log10(1e-5), !is.na(joint_z))     # smallest marginal P > 1e-5
tibble(n_weak = nrow(weak),
       n_joint_gws = sum(weak$joint_mlog10p > -log10(5e-8)),
       n_joint_1e5 = sum(weak$joint_mlog10p > -log10(1e-5)))

## representatives not genome-wide significant on their own (marginal P > 5e-8)
sub_gws <- cojo %>%
  filter(marginal_mlog10p < -log10(5e-8), !is.na(joint_z)) %>%
  mutate(change = factor(if_else(joint_mlog10p > marginal_mlog10p, "Stronger jointly", "Weaker jointly"),
                         levels = c("Stronger jointly", "Weaker jointly")))
sub_gws %>% count(change)
sub_gws %>% summarise(n = n(), joint_gws = sum(joint_mlog10p > -log10(5e-8)), joint_1e5 = sum(joint_mlog10p > -log10(1e-5)))

save_data(cojo,    "Fig6FG.AD_GWAS_COJO_Joint")
save_data(sub_gws, "Fig6FG.AD_GWAS_COJO_Joint_SubGWS")

n_pairs,n_cs,n_joint_missing
<int>,<int>,<int>
56,121,0


n_pairs,n_all_joint_below_1e5
<int>,<int>
56,56


n_weak,n_joint_gws,n_joint_1e5
<int>,<int>,<int>
11,6,11


change,n
<fct>,<int>
Stronger jointly,25
Weaker jointly,1


n,joint_gws,joint_1e5
<int>,<int>,<int>
26,16,26
